In [1]:
import torch
import transformers
import accelerate
import datasets

print("--- Environment Diagnostic Check ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Accelerate Version: {accelerate.__version__}")
print(f"Datasets Version: {datasets.__version__}")

# Verify Apple Silicon Hardware Binding
if torch.backends.mps.is_available():
    print("Hardware Status: M-Series GPU (MPS) is ACTIVE and ready.")
else:
    print("Hardware Status: WARNING - MPS not detected. Defaulting to CPU.")

/opt/anaconda3/envs/Master_Env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Environment Diagnostic Check ---
PyTorch Version: 2.6.0
Transformers Version: 5.2.0
Accelerate Version: 1.12.0
Datasets Version: 4.5.0
Hardware Status: M-Series GPU (MPS) is ACTIVE and ready.


In [2]:
import pandas as pd

# 1. Data Ingestion
# Note: Corrected the missing quote and updated to read_excel for .xlsx files.
file_path = '/Users/rakeshraushan/Ankit/JUPYTER_/spam_project/Final_data/final_spam_detection_data.xlsx'
df = pd.read_excel(file_path)

# 2. Data Profiling (Pre-Sanitization)
print("--- Initial Data Profiling ---")
print(f"Initial dataset shape: {df.shape}\n")

# A. Inspect Missing Values
missing_counts = df[['text', 'label']].isnull().sum()
print("Missing Value Counts by Column:")
print(f"{missing_counts}\n")

# B. Inspect Duplicates
duplicate_count = df.duplicated(subset=['text']).sum()
print(f"Total Duplicate Messages Identified: {duplicate_count}\n")

# C. Inspect Class Distribution (Visualizing the counts)
print("Class Label Distribution (Before Cleaning):")
print(f"{df['label'].value_counts(dropna=False)}\n")

# 3. Data Sanitization
print("--- Executing Data Sanitization ---")
df = df.dropna(subset=['text', 'label'])
df = df.drop_duplicates(subset=['text'])

# 4. Post-Sanitization Validation
print(f"Dataset shape after sanitization: {df.shape}\n")
print("Class Label Distribution (After Cleaning):")
print(f"{df['label'].value_counts()}\n")

# 5. Feature and Target Separation
X = df['text']
y = df['label']

--- Initial Data Profiling ---
Initial dataset shape: (505572, 2)

Missing Value Counts by Column:
text     0
label    0
dtype: int64

Total Duplicate Messages Identified: 226693

Class Label Distribution (Before Cleaning):
label
ham     254761
spam    250811
Name: count, dtype: int64

--- Executing Data Sanitization ---
Dataset shape after sanitization: (278879, 2)

Class Label Distribution (After Cleaning):
label
spam    206859
ham      72020
Name: count, dtype: int64



In [3]:
duplicate_count = df.duplicated(subset=['text']).sum()
print(f"Total Duplicate Messages Identified: {duplicate_count}\n")

Total Duplicate Messages Identified: 0



In [4]:
df.dtypes

text     object
label    object
dtype: object

In [5]:
df["label"].unique()


array(['ham', 'spam'], dtype=object)

In [6]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [7]:
label_counts=df['label'].value_counts().sort_index()
print(label_counts)

label
ham      72020
spam    206859
Name: count, dtype: int64


In [8]:
# 1. Separate the dataset by class
df_ham = df[df['label'] == 'ham']
df_spam = df[df['label'] == 'spam']

# 2. Identify the size of the minority class
minority_count = len(df_ham)

# 3. Randomly undersample the majority class (spam) to match the ham count
# We use a random_state for reproducibility
df_spam_downsampled = df_spam.sample(n=minority_count, random_state=42)

# 4. Concatenate the balanced dataframes back together
df_balanced = pd.concat([df_ham, df_spam_downsampled])

# 5. Shuffle the combined dataset to prevent sequential bias during training
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# 6. Validate the new distribution
print("Class Label Distribution (After Undersampling):")
print(f"{df_balanced['label'].value_counts()}\n")
print(f"New balanced dataset shape: {df_balanced.shape}")

Class Label Distribution (After Undersampling):
label
ham     72020
spam    72020
Name: count, dtype: int64

New balanced dataset shape: (144040, 2)


In [9]:
#Feature and Target Separation
X = df['text']
y = df["label"]


In [10]:
from sklearn.model_selection import train_test_split

# 3. Stratified Split
# test_size=0.20 reserves 20% of the data for evaluation
# random_state ensures reproducibility across different runs
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 223103 samples
Testing set size: 55776 samples


In [11]:
import pandas as pd
import re

# 1. Updated Text Standardization Function
def clean_text(text):
    # CRITICAL FIX: Cast the input to a string before applying .lower()
    text = str(text).lower()
    
    # Remove punctuation and special characters using regex
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace to keep the data tidy
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 2. Re-apply the function to the datasets
print("Re-applying text sanitization...")
X_train_cleaned = X_train.apply(clean_text)
X_test_cleaned = X_test.apply(clean_text)

print("Sanitization complete. Ready for TF-IDF Vectorization.")

Re-applying text sanitization...
Sanitization complete. Ready for TF-IDF Vectorization.


In [12]:
#%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [13]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from datasets import Dataset
import pandas as pd

print("--- Phase 4: Tokenization & Dataset Formatting ---")

# 1. Load the Pre-trained Tokenizer
print("Downloading and loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 2. Convert Pandas DataFrames to Hugging Face Datasets
# Assuming X_train_cleaned, X_test_cleaned, y_train, y_test are available from the previous split
train_df = pd.DataFrame({'text': X_train_cleaned, 'label': y_train})
test_df = pd.DataFrame({'text': X_test_cleaned, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 3. Define the Tokenization Function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256 # Reduced from 512 for faster processing of short SMS data
    )

# 4. Apply Tokenization in Batches
print("Tokenizing datasets in parallel batches...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Format the datasets to output PyTorch tensors
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("\n--- Phase 5: Neural Network Initialization ---")

# 5. Hardware Binding (Apple Silicon M-Series)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Hardware computing device selected: {device}")

# 6. Load the Base Model and map to hardware
print("Downloading DistilBERT architecture and configuring classification head...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=2 # Strictly configured for Spam (1) vs. Ham (0)
)

# Push the massive model architecture directly into the M4 Unified Memory
model.to(device)

print("\nPipeline setup complete. The model is resident in memory and ready for training.")

--- Phase 4: Tokenization & Dataset Formatting ---


Tokenizing datasets in parallel batches...


Map: 100%|██████████████████████████████████████████████████████████████| 55776/55776 [00:01<00:00, 34346.79 examples/s]



--- Phase 5: Neural Network Initialization ---
Hardware computing device selected: mps


Loading weights: 100%|█| 100/100 [00:00<00:00, 1966.99it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Pipeline setup complete. The model is resident in memory and ready for training.


In [16]:
import pandas as pd
from datasets import Dataset

print("--- Applying Label Encoding Correction ---")

# 1. Map string labels to binary integers
# Assuming X_train_cleaned, X_test_cleaned, y_train, y_test are still in memory
train_df = pd.DataFrame({'text': X_train_cleaned, 'label': y_train})
test_df = pd.DataFrame({'text': X_test_cleaned, 'label': y_test})

print("Transforming 'ham' to 0 and 'spam' to 1...")
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['label'].map({'ham': 0, 'spam': 1})

# 2. Re-create the Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 3. Re-apply Tokenization
print("Re-tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 4. Set Format to PyTorch Tensors
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("Correction applied successfully. Data is now pure numerical tensors.")

--- Applying Label Encoding Correction ---
Transforming 'ham' to 0 and 'spam' to 1...
Re-tokenizing datasets...


Map: 100%|██████████████████████████████████████████████████████████████| 55776/55776 [00:01<00:00, 35511.54 examples/s]

Correction applied successfully. Data is now pure numerical tensors.


In [17]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("--- Phase 6: Executing The Training Loop ---")

# 1. Define the Evaluation Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # Calculate precision, recall, and F1-score for binary classification
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 2. Configure Training Arguments for Apple Silicon (Updated for Transformers >= 4.46)
training_args = TrainingArguments(
    output_dir='./distilbert_checkpoints', 
    num_train_epochs=3,                    
    per_device_train_batch_size=16,        
    per_device_eval_batch_size=64,         
    learning_rate=2e-5,                    
    weight_decay=0.01,                     
    logging_dir='./training_logs',
    logging_steps=50,
    eval_strategy="epoch",                 # CRITICAL FIX: Renamed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,           
)

# 3. Initialize the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# 4. Initiate the Fine-Tuning Process
print("Starting training. The M4 chip is now processing the tensors...")
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


--- Phase 6: Executing The Training Loop ---
Starting training. The M4 chip is now processing the tensors...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.000086,0.001265,0.999821,0.999879,0.999976,0.999782
2,0.000000,0.002068,0.999821,0.999879,0.999927,0.999831
3,0.000028,0.001879,0.999821,0.999879,0.999952,0.999807


Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.96it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=41832, training_loss=0.0016585394370994973, metrics={'train_runtime': 29842.0414, 'train_samples_per_second': 22.428, 'train_steps_per_second': 1.402, 'total_flos': 4.433081106352435e+16, 'train_loss': 0.0016585394370994973, 'epoch': 3.0})

In [18]:
import os

print("--- Phase 7: Model and Tokenizer Serialization ---")

# 1. Define the Export Directory
# This creates a dedicated folder in your current working directory
export_directory = './production_spam_detector'

# Ensure the directory exists (creates it if it does not)
os.makedirs(export_directory, exist_ok=True)

# 2. Serialize the Neural Network Model
print(f"Saving fine-tuned model weights to: {export_directory}")
# This saves the model.safetensors and config.json files
trainer.save_model(export_directory)

# 3. Serialize the Preprocessing Artifact (The Tokenizer)
print(f"Saving tokenizer configurations to: {export_directory}")
# This saves the tokenizer.json, tokenizer_config.json, and vocab.txt files
tokenizer.save_pretrained(export_directory)

print("\nSerialization complete. All artifacts are securely stored on your hard drive.")

--- Phase 7: Model and Tokenizer Serialization ---
Saving fine-tuned model weights to: ./production_spam_detector


Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.46it/s]

Saving tokenizer configurations to: ./production_spam_detector

Serialization complete. All artifacts are securely stored on your hard drive.


In [19]:
import os

print("--- Phase 7: Final Evaluation & Artifact Serialization ---")

# 1. Final Unbiased Evaluation
print("Executing final evaluation against the 20% holdout test dataset...")
eval_results = trainer.evaluate()

print("\nFinal Model Performance Metrics:")
for key, value in eval_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').capitalize()
        print(f"{metric_name}: {value:.4f}")

# 2. Define the Export Directory
export_directory = './production_spam_detector'
os.makedirs(export_directory, exist_ok=True)

# 3. Serialize the Artifacts
print(f"\nSaving fine-tuned model weights (.safetensors) to: {export_directory}")
trainer.save_model(export_directory)

print(f"Saving tokenizer configurations to: {export_directory}")
tokenizer.save_pretrained(export_directory)

print("\nProject milestone achieved. All artifacts are securely stored on your hard drive.")

--- Phase 7: Final Evaluation & Artifact Serialization ---
Executing final evaluation against the 20% holdout test dataset...



Final Model Performance Metrics:
Loss: 0.0013
Accuracy: 0.9998
F1: 0.9999
Precision: 1.0000
Recall: 0.9998
Runtime: 577.8223
Samples_per_second: 96.5280
Steps_per_second: 1.5090

Saving fine-tuned model weights (.safetensors) to: ./production_spam_detector


Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.87it/s]

Saving tokenizer configurations to: ./production_spam_detector

Project milestone achieved. All artifacts are securely stored on your hard drive.


In [20]:
import torch
import transformers
import datasets
import evaluate
import accelerate
import pandas as pd
import numpy as np

print("--- Core Library Versions for Deployment ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Datasets Version: {datasets.__version__}")
print(f"Evaluate Version: {evaluate.__version__}")
print(f"Accelerate Version: {accelerate.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

--- Core Library Versions for Deployment ---
PyTorch Version: 2.6.0
Transformers Version: 5.2.0
Datasets Version: 4.5.0
Evaluate Version: 0.4.6
Accelerate Version: 1.12.0
Pandas Version: 2.3.3
NumPy Version: 2.2.6
